# Prompt Experimentation

# LLM Sampling Parameters Explained

## Temperature

Temperature controls the **randomness/creativity** of the model's output by adjusting how "peaked" or "flat" the probability distribution over next tokens is.

- **Low temperature (e.g., 0–0.3)**: Model becomes more deterministic, favoring high-probability words. Good for factual, consistent tasks.
- **High temperature (e.g., 0.8–1.5)**: Flattens the distribution, giving lower-probability words a better chance. Output becomes more creative/varied but riskier (can get incoherent).

**How it works internally:** temperature divides the logits before softmax. Lower temperature sharpens differences between token probabilities; higher temperature smooths them out.

**Example prompt:** "The cat sat on the ___"

| Temperature | Likely output |
|---|---|
| 0.1 | "mat" (very predictable, almost always the same) |
| 0.7 | "mat", "chair", "windowsill" (some variety) |
| 1.3 | "moon", "keyboard", "philosophy textbook" (wild, less coherent) |

**When to use what:**
- Code generation, math, factual Q&A → low temperature (0–0.3)
- Creative writing, brainstorming, poetry → higher temperature (0.7–1.2)

---

## Top-p (Nucleus Sampling)

Top-p controls randomness differently — instead of reshaping the whole distribution, it **restricts the pool of tokens the model can choose from**, to only those whose cumulative probability adds up to *p*.

- **Top-p = 1.0**: Consider all possible tokens (no restriction).
- **Top-p = 0.9**: Only consider the smallest set of tokens whose combined probability is ≥ 90%. Rare/unlikely tokens get cut out entirely.
- **Top-p = 0.5**: Even more restrictive — only the most probable core of tokens is eligible.

**Example:** Suppose next-token probabilities are:
```
"mat"    → 40%
"chair"  → 25%
"floor"  → 15%
"moon"   → 8%
"idea"   → 5%
...(long tail)... 
```

- With **top-p = 0.9**, the model keeps sampling from {"mat", "chair", "floor", "moon"} (cumulative ~90%) and discards the long tail.
- With **top-p = 0.5**, it keeps only {"mat", "chair"} (~65%, first set that crosses 50%... roughly), narrowing choices further.

**Key difference from temperature:** Temperature reshapes *how sharp* the distribution is; top-p *cuts off* the tail of unlikely options entirely. They're often used together — e.g., temperature = 0.7, top-p = 0.9 is a common combo that adds some randomness while avoiding truly bizarre tokens.

---

## Max Tokens

This is simply a **hard limit on the length of the output** the model is allowed to generate (measured in tokens, not words or characters — a token is roughly ¾ of a word in English).

- **Max tokens = 50**: The response will be cut off after ~50 tokens, even mid-sentence, if the model hasn't naturally finished.
- **Max tokens = 1000**: Allows for a much longer, more detailed response.

**Example:**

Prompt: "Explain quantum computing."

- `max_tokens = 20` → *"Quantum computing uses qubits, which can exist in superpositions of 0 and 1, unlike classical"* (cuts off abruptly)
- `max_tokens = 300` → A full multi-paragraph explanation with examples and a conclusion.

**Practical note:** This doesn't control *quality*, only *quantity*. It's mainly used to control cost, latency, or to prevent runaway generations (e.g., in a chat UI where you don't want a 5,000-word reply to a simple question).

---

## Putting It Together

A typical config for different use cases:

| Use case | Temperature | Top-p | Max tokens |
|---|---|---|---|
| Code generation | 0.2 | 0.95 | 1024 |
| Customer support chatbot | 0.3 | 0.9 | 300 |
| Creative story writing | 0.9 | 0.95 | 2000 |
| Brainstorming ideas | 1.0 | 1.0 | 500 |

**Rule of thumb:** Adjust *either* temperature *or* top-p to control randomness — changing both simultaneously in extreme directions makes behavior harder to predict. Max tokens is more about resource/output-length control than about creativity or accuracy.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

# LOW temperature - deterministic, predictable
response_low = client.responses.create(
    model="gpt-4.1",
    input="The cat sat on the",
    temperature=0.1
)
print("Low temp (0.1):", response_low.output_text)

# MEDIUM temperature - balanced
response_medium = client.responses.create(
    model="gpt-4.1",
    input="The cat sat on the",
    temperature=0.7
)
print("Medium temp (0.7):", response_medium.output_text)

# High temperature - creative/random
response_high = client.responses.create(
    model="gpt-4.1",
    input="The cat sat on the",
    temperature=1.5
)
print("Low temp (1.5):", response_high.output_text)

# High temperature - creative/random
response_highest = client.responses.create(
    model="gpt-4.1",
    input="The cat sat on the",
    temperature=2
)
print("Highest temp (2):", response_highest.output_text)


Low temp (0.1): mat.

Would you like to continue the story or do something else with this sentence?
Medium temp (0.7): ...mat.

This is a famous example of a simple English sentence often used to demonstrate basic sentence structure and teach reading to children! If you’d like to continue the sentence or explore creative variations, let me know!
Low temp (1.5): ...mat!

This is a classic phrase often used to demonstrate simple English sentence structure:  
**“The cat sat on the mat.”**

Would you like me to expand on this new classic tale or help with something else? 😺
Highest temp (2): mat.

(This is the most common version of that famous sentence!)

Want to expand or get creative? Here's a rendition that might amuse you:

*The cat sat on the playwright in the fourth act clutching an applicable grammar handbook!*

Or—if you'd like the sentence to keep illustrating English-phase adaptability, you can play with it forever! Classic choices could be:

- "The cat sat on the human."
- "The 

In [ ]:
prompt = "Write one sentence about the future of technology."

# LOW top_p - narrow, safe token pool
response_narrow = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    top_p=0.1,       # only most probable tokens considered
    temperature=1.0, # keep temp neutral to isolate top_p effect
    max_tokens=50
)
print("top_p=0.1:", response_narrow.choices[0].message.content)

# HIGH top_p - wide, diverse token pool
response_wide = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    top_p=0.95,
    temperature=1.0,
    max_tokens=50
)
print("top_p=0.95:", response_wide.choices[0].message.content)

top_p=0.1: The future of technology promises unprecedented advancements in artificial intelligence, quantum computing, and biotechnology, fundamentally transforming how we live, work, and interact with the world.
top_p=0.95: The future of technology promises unprecedented advancements in artificial intelligence, quantum computing, and biotechnology, fundamentally transforming how we live and interact with the world.


In [ ]:
prompt = "Explain how photosynthesis works."

# SHORT response - cut off quickly
response_short = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=15,
    temperature=0.5
)
print("max_tokens=15:", response_short.choices[0].message.content)
print("Finish reason:", response_short.choices[0].finish_reason)  # likely "length"

# LONG response - full explanation allowed
response_long = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500,
    temperature=0.5
)
print("max_tokens=500:", response_long.choices[0].message.content)
print("Finish reason:", response_long.choices[0].finish_reason)  # likely "stop"

max_tokens=15: Photosynthesis is a biochemical process by which green plants, algae, and certain
Finish reason: length
max_tokens=500: Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy, usually from the sun, into chemical energy stored in glucose. This process is fundamental to life on Earth as it provides the primary energy source for nearly all ecosystems. Here's how it works:

1. **Light Absorption**: Photosynthesis begins when chlorophyll, the green pigment in plant cells, absorbs sunlight. Chlorophyll is located in the chloroplasts, primarily within the leaves of plants.

2. **Water Splitting (Photolysis)**: The absorbed light energy is used to split water molecules (H₂O) into oxygen, protons, and electrons. This occurs in the thylakoid membranes of the chloroplasts. The oxygen is released as a byproduct into the atmosphere.

3. **Electron Transport Chain**: The electrons released from the water molecules are transferred through a s

## Combined Example (All Three Together)

In [ ]:
def generate_response(prompt, temperature=0.7, top_p=1.0, max_tokens=200):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )
    return {
        "text": response.choices[0].message.content,
        "finish_reason": response.choices[0].finish_reason,
        "tokens_used": response.usage.completion_tokens
    }

# Use case 1: Code generation (deterministic, controlled length)
code_config = generate_response(
    prompt="Write a Python function to reverse a string.",
    temperature=0.2,
    top_p=0.95,
    max_tokens=100
)
print(code_config)

# Use case 2: Creative story (high randomness, longer output)
story_config = generate_response(
    prompt="Write a short sci-fi story opening.",
    temperature=0.95,
    top_p=1.0,
    max_tokens=400
)
print(story_config)

# Use case 3: Customer support (safe, concise, on-topic)
support_config = generate_response(
    prompt="How do I reset my password?",
    temperature=0.3,
    top_p=0.9,
    max_tokens=150
)
print(support_config)

{'text': 'Certainly! You can reverse a string in Python using a simple function. Here\'s one way to do it:\n\n```python\ndef reverse_string(s):\n    """\n    This function takes a string as input and returns the reversed string.\n    \n    :param s: The string to be reversed.\n    :return: The reversed string.\n    """\n    return s[::-1]\n\n# Example usage:\noriginal_string = "Hello, World!"\nreversed_string = reverse_string(original_string)\nprint(reversed_string) ', 'finish_reason': 'length', 'tokens_used': 100}
{'text': 'The metallic hum of the interstellar freighter, the Celestial Echo, reverberated through the deep, silent void as it cruised through the uncharted expanse of the Zoradian Nebula. Captain Mira Cole stood on the bridge, gazing through the ship\'s viewport at the swirling clouds of iridescent gas and distant, pulsating stars. This region of space was both breathtaking and eerie, a place where the very laws of physics seemed to whisper their defiance.\n\n"We\'re approa

## Streaming Example (useful when max_tokens is large)

In [ ]:

stream = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Write a 500-word essay on climate change."}],
    temperature=0.7,
    top_p=0.9,
    max_tokens=700,
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

**Title: The Urgency of Climate Change: A Call to Action**

Climate change, an existential challenge of our time, is a complex phenomenon that refers to long-term alterations in temperature, precipitation, wind patterns, and other elements of the Earth's climate system. Driven largely by anthropogenic activities, primarily the burning of fossil fuels and deforestation, climate change poses a significant threat to the planet's ecosystems, biodiversity, and human societies.

The scientific consensus is clear: human activities have increased concentrations of greenhouse gases (GHGs) in the atmosphere, leading to global warming and climate disruptions. Carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O) are the primary GHGs responsible for trapping heat in the atmosphere, a process known as the greenhouse effect. Since the Industrial Revolution, CO2 levels have risen from approximately 280 parts per million (ppm) to over 410 ppm today, a concentration not seen in millions of years